#**In LangChain, the Agent can access many dataframes at once**

#**01: Install All the Required Packages**

In [1]:
!pip install langchain langchain_experimental
!pip install watermark
!pip install langchain_openai

  Using cached langchain_core-0.3.84-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 4.8 MB/s eta 0:00:0000:01
Using cached langchain_core-0.3.84-py3-none-any.whl (459 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.0
    Uninstalling langchain-core-1.3.0:
      Successfully uninstalled langchain-core-1.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.4 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.84 which is incompatible.
langchain-classic 1.0.4 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
la

#**02: Import All The Required Libraries**

In [2]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings("ignore")

In [3]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import OpenAI

#**03: Setup the Environment**


In [4]:
os.environ['OPENAI_API_KEY'] =  getpass("OpenAI API Key: ")

#**04:Data**

In [6]:
import pandas as pd

In [7]:
url = "https://raw.githubusercontent.com/adamerose/datasets/master/titanic.csv"
df=pd.read_csv(url)
print(df.shape)
df.head()

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [12]:
df.shape

(891, 15)

#**05: Single DataFrame Example**

#**Agent can interact with Single DataFrame**

In [8]:
llm=OpenAI()

In [9]:
agent=create_pandas_dataframe_agent(llm, df, verbose=True, allow_dangerous_code=True)

In [11]:
agent.run("How many rows are there")

/tmp/ipykernel_114199/1005132620.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent.run("How many rows are there")




> Entering new AgentExecutor chain...
Thought: I need to use the len() function to find the number of rows in the dataframe
Action: python_repl_ast
Action Input: len(df)891891 rows seems like a reasonable number for a dataset
Final Answer: 891 rows

> Finished chain.


'891 rows'

In [13]:
agent.run("How many people have more than 23 age")



> Entering new AgentExecutor chain...
Thought: We need to filter the dataframe by age and then count the number of rows
Action: python_repl_ast
Action Input: df[df['age']>23]     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
1           1       1  female  38.0      1      0  71.2833        C   First   
2           1       3  female  26.0      0      0   7.9250        S   Third   
3           1       1  female  35.0      1      0  53.1000        S   First   
4           0       3    male  35.0      0      0   8.0500        S   Third   
6           0       1    male  54.0      0      0  51.8625        S   First   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
884         0       3    male  25.0      0      0   7.0500        S   Third   
885         0       3  female  39.0      0      5  29.1250        Q   Third   
886         0       2    male  27.0      0      0  13.0000        S  Second   
889         1       1    male  26.

'468'

#**06: Multi DataFrame Example**

#**Agent can also interact with Multiple DataFrames passed in a list**

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


In [15]:
df1=df.copy()

In [16]:
df1

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [17]:
df1["age"] = df1["age"].fillna(df1["age"].mean())

In [18]:
agent=create_pandas_dataframe_agent(llm, [df, df1], verbose=True,allow_dangerous_code=True)

In [19]:
agent.run("How many rows in the age column are different")



> Entering new AgentExecutor chain...
Thought: I should find the number of unique values in the age column
Action: python_repl_ast
Action Input: len(df['age'].unique())NameError: name 'df' is not definedI need to assign the dataframes to variables before using them
Action: python_repl_ast
Action Input: df1 = df.head()NameError: name 'df' is not definedI need to import pandas to use dataframes
Action: python_repl_ast
Action Input: import pandas as pdI can now use the dataframes with the assigned variable names
Action: python_repl_ast
Action Input: len(df1['age'].unique())8989 unique values in the age column means there are 89 different rows
Final Answer: 89 rows in the age column are different

> Finished chain.


'89 rows in the age column are different'

In [20]:
df2=df1.copy()

In [21]:
df2["Age_Multiplied"]=df1["age"] * 2

In [22]:
df2.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,Age_Multiplied
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,44.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,76.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,52.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,70.0
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,70.0


In [23]:
agent=create_pandas_dataframe_agent(llm, [df,df1, df2], verbose=True, allow_dangerous_code=True)

In [24]:
agent.run("Are the number of columns same in all the dataframe")



> Entering new AgentExecutor chain...
Thought: I can use the `columns` attribute of the dataframe to check the number of columns.
Action: [python_repl_ast]
Action Input: df.columns[python_repl_ast] is not a valid tool, try one of [python_repl_ast]. I can use the `columns` attribute of the dataframe to check the number of columns.
Action: [python_repl_ast]
Action Input: df1.columns[python_repl_ast] is not a valid tool, try one of [python_repl_ast]. I can use the `columns` attribute of the dataframe to check the number of columns.
Action: [python_repl_ast]
Action Input: df2.columns[python_repl_ast] is not a valid tool, try one of [python_repl_ast]. I can use the `columns` attribute of the dataframe to check the number of columns.
Action: [python_repl_ast]
Action Input: df3.columns[python_repl_ast] is not a valid tool, try one of [python_repl_ast]. I can use the `columns` attribute of the dataframe to check the number of columns.
Action: [python_repl_ast]
Action Input: len(df1.columns) 

'The number of columns is the same in all the dataframes.'